# 07-units-O3in -- Dimensionally constrained PM2.5 search with ozone in the pool (Bogota)

Follows `06_pm25_with_o3_feature_selection.ipynb`: when ozone (MDA8) is offered to the PM2.5 feature selection it is
chosen at step 2. This notebook runs the same constrained PySR search as every other constrained run in the paper (same
operators, `niterations=200`, `populations=24`, `maxsize=25`, three seeds, units: gases and PM `kg/m^3`, wind `m/s`,
`w700` `Pa/s`, fires dimensionless) on the selected variables, without the calendar term (the paper reports no calendar terms).
The rows are the days with an ozone value. The rows labelled `free-form` in the scored table are the *lead PM2.5 front
without ozone* (`bogota_pm25_pysr_units_pareto_front.csv`), re-scored by the same leave-one-year-out procedure on the same rows.
Ozone and PM2.5 are each other's targets, so an equation that uses ozone to predict PM2.5 is a sensitivity test of the coupling, not a
replacement for Eq. 1 (it would make the two equations circular).

In [1]:
# %% Settings (edit here, then Run All)
CITY, TARGET, TAG = "bogota", "pm25", "units_o3_fire2"
MASTER = "bogota_master_dataset_with_o3.parquet"      # PM2.5 master + the paper's ozone (MDA8) and its lags
SELECTION_JSON = "bogota_pm25_o3_pool.json"            # sfs-with-ozone selection, calendar term removed
EXTRA_FEATURES = []                                    # none added
DROP_FEATURES = []                                     # none removed
SEEDS = [0, 1, 2]
PYSR_KW = dict(niterations=200, binary_operators=["+", "-", "*", "/"], unary_operators=["log", "sqrt", "exp"],
               model_selection="best", maxsize=25, populations=24)     # identical to every constrained run in the paper
UNIT_MAP = {"co": "kg/m^3", "o3_mda8": "kg/m^3", "o3_mda8_lag1": "kg/m^3", "wind_speed_era5": "m/s", "hotspots_sector_E_lag2": "", "hotspots_magdalena_valley_lag2": "", "tcwv": "kg/m^2", "tke_proxy": "m^2/s^2"}
Y_UNITS = "kg/m^3"
FREEFORM_FRONT = "bogota_pm25_pysr_units_pareto_front.csv"   # the lead PM2.5 front (no ozone), scored on the SAME rows: label 'free-form' in the table = this front
MAX_COMPLEXITY_REPORTED = 25

In [2]:
import sys, json, re, warnings
from pathlib import Path
PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))
import numpy as np, pandas as pd, sympy as sp
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / CITY
df = pd.read_parquet(DATA_DIR / MASTER)
df["date"] = pd.to_datetime(df["date"]); df["year"] = df["date"].dt.year
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
selection = json.load(open(DATA_DIR / SELECTION_JSON))
features = [f for f in selection["selected_no_lag"] if f not in DROP_FEATURES]
for f in EXTRA_FEATURES:
    if f not in features: features.append(f)
print("pool:", features)
data = df.dropna(subset=features + [TARGET]).reset_index(drop=True)
X = data[features].to_numpy(); y = data[TARGET].to_numpy()
X_units = [UNIT_MAP[f] for f in features]
print(data.shape, dict(zip(features, X_units)), "years:", sorted(data.year.unique()))

pool: ['co', 'o3_mda8', 'hotspots_sector_E_lag2', 'wind_speed_era5', 'hotspots_magdalena_valley_lag2', 'tcwv', 'tke_proxy', 'o3_mda8_lag1']
(3281, 220) {'co': 'kg/m^3', 'o3_mda8': 'kg/m^3', 'hotspots_sector_E_lag2': '', 'wind_speed_era5': 'm/s', 'hotspots_magdalena_valley_lag2': '', 'tcwv': 'kg/m^2', 'tke_proxy': 'm^2/s^2', 'o3_mda8_lag1': 'kg/m^3'} years: [np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022)]


In [3]:
from pysr import PySRRegressor
fronts = {}
for seed in SEEDS:
    sr = PySRRegressor(**PYSR_KW, random_state=seed)
    sr.fit(X, y, variable_names=features, X_units=X_units, y_units=Y_UNITS)
    f = sr.equations_[["complexity", "loss", "equation"]].copy(); f["seed"] = seed
    f.to_csv(DATA_DIR / f"{CITY}_{TARGET}_pysr_{TAG}_pareto_front_seed{seed}.csv", index=False)
    fronts[seed] = f
    print("seed", seed, "done; best:", sr.get_best()["equation"])

Compiling Julia backend...


/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(



Expressions evaluated per second: 8.030e+03
Progress: 20 / 200 total iterations (10.000%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.181e+01  0.000e+00  y[m⁻³ kg] = o3_mda8[m⁻³ kg]
3           3.990e+01  2.189e-01  y[m⁻³ kg] = co[m⁻³ kg] * 23.839[?]
5           3.234e+01  1.050e-01  y[m⁻³ kg] = (o3_mda8[m⁻³ kg] * 1.1294[?]) * co[m⁻³...
                                               kg]
7           2.589e+01  1.112e-01  y[m⁻³ kg] = ((o3_mda8[m⁻³ kg] * 0.78688[?]) + 7.60...
                                              27[?]) * co[m⁻³ kg]
9           2.362e+01  4.587e-02  y[m⁻³ kg] = co[m⁻³ kg] * (((o3_mda8_lag1[m⁻³ kg] *...
                                               o3_mda8[m⁻³ kg]) * 0.016499[?]) + 16.399[?])
10          2.168e+01  8.550e-02  y[m⁻³ kg] = (sqrt(hotspots_

┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


seed 0 done; best: (co * 18.757557) + (o3_mda8 * (sqrt(hotspots_sector_E_lag2) * 0.012707745))


/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(



Expressions evaluated per second: 9.690e+03
Progress: 24 / 200 total iterations (12.125%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.181e+01  0.000e+00  y[m⁻³ kg] = o3_mda8[m⁻³ kg]
3           3.990e+01  2.189e-01  y[m⁻³ kg] = co[m⁻³ kg] * 23.851[?]
5           3.242e+01  1.037e-01  y[m⁻³ kg] = (o3_mda8[m⁻³ kg] * co[m⁻³ kg]) * 1.146...
                                              6[?]
6           2.665e+01  1.961e-01  y[m⁻³ kg] = co[m⁻³ kg] * sqrt(o3_mda8[m⁻³ kg] * 29...
                                              .45[?])
7           2.550e+01  4.408e-02  y[m⁻³ kg] = (o3_mda8[m⁻³ kg] + 13.956[?]) * (co[m⁻...
                                              ³ kg] * 0.69255[?])
9           2.548e+01  3.455e-04  y[m⁻³ kg] = (((o3_mda8[m⁻³ kg] + 13.87[?]) * 0.692...
           

┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:
/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(



Expressions evaluated per second: 9.370e+03
Progress: 23 / 200 total iterations (11.938%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.181e+01  0.000e+00  y[m⁻³ kg] = o3_mda8[m⁻³ kg]
3           3.990e+01  2.189e-01  y[m⁻³ kg] = co[m⁻³ kg] * 23.813[?]
5           3.274e+01  9.889e-02  y[m⁻³ kg] = (co[m⁻³ kg] * o3_mda8[m⁻³ kg]) * 1.166...
                                              6[?]
6           2.665e+01  2.057e-01  y[m⁻³ kg] = (sqrt(o3_mda8[m⁻³ kg]) * co[m⁻³ kg]) *...
                                               5.3858[?]
7           2.547e+01  4.542e-02  y[m⁻³ kg] = ((o3_mda8[m⁻³ kg] + 14.305[?]) * co[m⁻...
                                              ³ kg]) * 0.69092[?]
9           2.545e+01  3.998e-04  y[m⁻³ kg] = (co[m⁻³ kg] * ((o3_mda8[m⁻³ kg] + 13.7...
        

┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


In [4]:
FLOAT = re.compile(r"(?<![A-Za-z_0-9.])\d+\.\d+(?:[eE][+-]?\d+)?")
def make_model(eq, feats):
    vals = []
    def sub(m): vals.append(float(m.group(0))); return f"c{len(vals)-1}"
    s = FLOAT.sub(sub, eq)
    syms = {f: sp.Symbol(f) for f in feats}; cs = [sp.Symbol(f"c{i}") for i in range(len(vals))]
    expr = sp.sympify(s, locals={**syms, **{str(c): c for c in cs}, "sqrt": sp.sqrt, "log": sp.log, "exp": sp.exp})
    used = [f for f in feats if syms[f] in expr.free_symbols]
    keep = [i for i, c in enumerate(cs) if c in expr.free_symbols]
    fn = sp.lambdify([syms[f] for f in used] + [cs[i] for i in keep], expr, "numpy")
    p0 = np.array([vals[i] for i in keep])
    return (lambda Xt, *p: fn(*Xt, *p)), used, p0

yy = data[TARGET].to_numpy(float); years = data["year"].to_numpy(); YRS = sorted(set(years))
def score(eq, feats):
    model, used, p0 = make_model(eq, feats)
    if not used:                                   # constant equation: leave-one-year-out mean
        oof = np.concatenate([np.full((years == yr).sum(), np.mean(yy[years != yr])) for yr in YRS]); yo = np.concatenate([yy[years == yr] for yr in YRS])
        per = [r2_score(yy[years == yr], np.full((years == yr).sum(), np.mean(yy[years != yr]))) for yr in YRS]
        return dict(r2_insample=0.0, r2_pooled=r2_score(yo, oof), r2_mean_years=float(np.mean(per)), vars="")
    Xt = tuple(data[f].to_numpy(float) for f in used)
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try: popt, _ = curve_fit(model, Xt, yy, p0=p0, maxfev=20000)
        except Exception: popt = p0
        pred = np.full(len(yy), np.nan)
        for yr in YRS:
            te = years == yr
            try: pf, _ = curve_fit(model, tuple(a[~te] for a in Xt), yy[~te], p0=popt, maxfev=20000)
            except Exception: pf = popt
            pred[te] = model(tuple(a[te] for a in Xt), *pf)
        ok = np.all(np.isfinite(pred)); ins = model(Xt, *popt)
        per = [r2_score(yy[years == yr], pred[years == yr]) for yr in YRS] if ok else [np.nan]
        return dict(r2_insample=r2_score(yy, ins) if np.all(np.isfinite(ins)) else np.nan, r2_pooled=r2_score(yy, pred) if ok else np.nan,
                    r2_mean_years=float(np.mean(per)), vars=",".join(used))

rows = []
for seed, f in fronts.items():
    for _, r in f.iterrows():
        if r.complexity <= MAX_COMPLEXITY_REPORTED: rows.append(dict(seed=str(seed), complexity=int(r.complexity), equation=r.equation, **score(r.equation, features)))
ff = pd.read_csv(DATA_DIR / FREEFORM_FRONT).sort_values("complexity")
ff_feats = sorted({t for e in ff.equation for t in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", e) if t in data.columns})
for _, r in ff.iterrows():
    if r.complexity <= MAX_COMPLEXITY_REPORTED:
        try: rows.append(dict(seed="free-form", complexity=int(r.complexity), equation=r.equation, **score(r.equation, ff_feats)))
        except Exception as e: print("free-form eq skipped", r.complexity, e)
scored = pd.DataFrame(rows); scored.to_csv(DATA_DIR / f"{CITY}_{TARGET}_pysr_{TAG}_scored.csv", index=False)
pd.set_option("display.width", 250, "display.max_colwidth", 110)
print("n days", len(data)); print(scored[["seed", "complexity", "r2_pooled", "r2_mean_years", "vars"]].round(3).to_string(index=False))

n days 3281
     seed  complexity  r2_pooled  r2_mean_years                                                                                          vars
        0           1      0.041         -0.077                                                                                       o3_mda8
        0           3      0.370          0.341                                                                                            co
        0           5      0.483          0.438                                                                                    co,o3_mda8
        0           6      0.576          0.552                                                                                    co,o3_mda8
        0           7      0.589          0.563                                                                                    co,o3_mda8
        0           9      0.619          0.590                                                                       co,o3_mda8,o3_mda8